# 04 — Full Pipeline Run

This notebook runs the complete neoantigen prediction pipeline end-to-end on the HCC1395 breast cancer cell line data.

**Pipeline steps:**
1. Load HLA types from OptiType output
2. Parse missense variants from VEP-annotated VCF
3. Filter by gene expression
4. Load reference proteome
5. Generate peptide candidates
6. MHC-I binding/presentation prediction (MHCflurry)
7. Compute agretopicity scores
8. Rank by composite score

Data: `data/HCC1395_inputs/`

> **Note**: MHCflurry models must be downloaded before running step 6.  
> Run `mhcflurry-downloads fetch` once in your terminal if you haven't already.

In [ ]:
import logging
import sys
import time
import pprint
import dataclasses
from pathlib import Path

import pandas as pd

# Add src to path if running from notebooks directory
sys.path.insert(0, str(Path("../src").resolve()))

from neoantigen_pipeline import NeoantigenPipeline
from neoantigen_pipeline.config import PipelineConfig

# Enable per-step timing: the pipeline logs each step with elapsed time via
# the neoantigen_pipeline logger at INFO level. Configuring the handler here
# makes those messages visible in the notebook output.
logging.basicConfig(
    level=logging.WARNING,  # suppress noisy third-party logs
    format="%(message)s",
)
_pl_logger = logging.getLogger("neoantigen_pipeline")
_pl_logger.setLevel(logging.INFO)
if not _pl_logger.handlers:
    _pl_handler = logging.StreamHandler(sys.stdout)
    _pl_handler.setFormatter(logging.Formatter("%(message)s"))
    _pl_logger.addHandler(_pl_handler)

print("Imports OK")

In [2]:
# Define all data paths
DATA_DIR      = Path("../data/HCC1395_inputs")
VCF_PATH      = DATA_DIR / "annotated.expression.vcf.gz"
HLA_PATH      = DATA_DIR / "optitype_normal_result.tsv"
PROTEOME_PATH = DATA_DIR / "Homo_sapiens.GRCh38.pep.all.fa.gz"
CONFIG_PATH   = Path("../configs/default.yaml")

print("Path availability:")
for name, p in [("VCF", VCF_PATH), ("HLA", HLA_PATH),
                ("Proteome", PROTEOME_PATH), ("Config", CONFIG_PATH)]:
    print(f"  {name:10s}: {p.exists()}  ({p.resolve()})")

Path availability:
  VCF       : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/annotated.expression.vcf.gz)
  HLA       : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/optitype_normal_result.tsv)
  Proteome  : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/data/HCC1395_inputs/Homo_sapiens.GRCh38.pep.all.fa.gz)
  Config    : True  (/home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/configs/default.yaml)


## Pipeline Configuration

In [3]:
# Load pipeline configuration from YAML
config = PipelineConfig.from_yaml(str(CONFIG_PATH))

print("PipelineConfig:")
try:
    pprint.pprint(dataclasses.asdict(config), indent=2)
except TypeError:
    # Fallback if config is not a dataclass
    pprint.pprint(vars(config) if hasattr(config, "__dict__") else str(config), indent=2)

PipelineConfig:
{ 'expression_filter': {'filter_missing': False, 'min_expression': 1.0},
  'mhc_i': { 'alleles': ( 'HLA-A*29:02',
                          'HLA-B*45:01',
                          'HLA-B*82:02',
                          'HLA-C*06:02'),
             'binding_affinity_threshold_nm': 500.0,
             'peptide_lengths': (8, 9, 10, 11),
             'percentile_rank_threshold': 2.0,
             'use_presentation_score': True},
  'output_dir': 'results',
  'peptide_generation': { 'c_flank_length': 10,
                          'n_flank_length': 10,
                          'peptide_lengths': (8, 9, 10, 11)},
  'scoring': { 'agretopicity_weight': 0.2,
               'expression_weight': 0.2,
               'presentation_score_weight': 0.4,
               'vaf_weight': 0.2}}


## Running the Pipeline

The `NeoantigenPipeline.run()` method executes all steps in sequence.  
MHCflurry prediction is the most time-consuming step.

In [4]:
pipeline = NeoantigenPipeline(config)

print("Starting pipeline run ...")
t0 = time.time()
results = pipeline.run(
    str(VCF_PATH),
    str(HLA_PATH),
    str(PROTEOME_PATH),
)
elapsed = time.time() - t0

print(f"Pipeline completed in {elapsed:.1f}s")

Starting pipeline run ...


TXNDC15_p.Ser7Pro: filtered 4 peptide candidate(s) containing non-standard amino acids (kept 24)
I0000 00:00:1774280369.774927  180815 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774280369.813154  180815 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774280371.176909  180815 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
E0000 00:00:1774280375.513816  180815

Predicting processing.


  0%|          | 0/1 [00:00<?, ?it/s]

4/4 [==============================] - 1s 227ms/step


...

4/4 [==============================] - 4s 1s/step


 75%|███████▌  | 3/4 [00:13<00:04,  4.56s/it]

4/4 [==============================] - 4s 1s/step


100%|██████████| 4/4 [00:18<00:00,  4.55s/it]


Pipeline completed in 72.3s


## Results Overview

In [5]:
# Convert results to DataFrame for inspection
df = results.to_dataframe()
print(f"Total neoantigen candidates: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
display(df.head(20))

Total neoantigen candidates: 14240
Columns: ['gene', 'mutation', 'peptide', 'wildtype_peptide', 'best_allele', 'presentation_score', 'binding_affinity_nm', 'wildtype_affinity_nm', 'processing_score', 'agretopicity', 'expression', 'vaf', 'composite_score', 'composite_rank']



,gene,mutation,peptide,wildtype_peptide,best_allele,presentation_score,binding_affinity_nm,wildtype_affinity_nm,processing_score,agretopicity,expression,vaf,composite_score,composite_rank
0,TESK1,TESK1_p.His539Tyr,YSLPRAAAL,HSLPRAAAL,HLA-C*06:02,0.977559,50.918984,67.006494,0.901196,1.315943,3.54861,1.00000,0.601534,1
1,TESK1,TESK1_p.His539Tyr,YSLPRAAAL,HSLPRAAAL,HLA-C*06:02,0.977559,50.918984,67.006494,0.901196,1.315943,3.54861,1.00000,0.601534,2
2,DDX3X,DDX3X_p.Arg108Thr,STVRPCVVY,SRVRPCVVY,HLA-A*29:02,0.948855,59.585606,158.463266,0.716968,2.659422,35.65300,0.96552,0.594814,3
3,MED14,MED14_p.Phe1325Leu,LLPDQATQL,LFPDQATQL,HLA-C*06:02,0.954728,60.241104,52.460411,0.753466,0.870841,7.17315,1.00000,0.593279,4
4,PRDX5,PRDX5_p.Phe157Leu,LLADPTGAL,LLADPTGAF,HLA-C*06:02,0.763804,351.942994,172.438166,0.712343,0.489961,549.85300,0.42745,0.591919,5
5,MED14,MED14_p.Phe224Leu,LLPDQATQL,LFPDQATQL,HLA-C*06:02,0.954728,60.241104,52.460411,0.753466,0.870841,1.43694,1.00000,0.591189,6
6,ZNF548,ZNF548_p.Asp24Tyr,TQGRVVFEY,TQGRVVFED,HLA-A*29:02,0.737393,43.418451,18744.164202,0.139353,431.709652,5.41922,0.46847,0.591101,7
7,ZNF548,ZNF548_p.Asp15Tyr,TQGRVVFEY,TQGRVVFED,HLA-A*29:02,0.737393,43.418451,18744.164202,0.139353,431.709652,1.55660,0.46847,0.589693,8
8,DDX3X,DDX3X_p.Arg261Thr,STVRPCVVY,SRVRPCVVY,HLA-A*29:02,0.948855,59.585606,158.463266,0.716968,2.659422,9.07993,0.96552,0.585131,9
9,DDX3X,DDX3X_p.Arg294Thr,STVRPCVVY,SRVRPCVVY,HLA-A*29:02,0.948855,59.585606,158.463266,0.716968,2.659422,8.88749,0.96552,0.585061,10


In [6]:
# Show top 10 candidates with key scoring columns
key_cols = ["gene", "mutation", "peptide", "best_allele",
            "presentation_score", "agretopicity",
            "composite_score", "composite_rank"]

# Use only columns that exist in the dataframe
available_cols = [c for c in key_cols if c in df.columns]
missing_cols   = [c for c in key_cols if c not in df.columns]

if missing_cols:
    print(f"Note: these expected columns were not found and will be skipped: {missing_cols}")
    print(f"Available columns: {list(df.columns)}")

print("\nTop 10 neoantigen candidates:")
display(df[available_cols].head(10))


Top 10 neoantigen candidates:


,gene,mutation,peptide,best_allele,presentation_score,agretopicity,composite_score,composite_rank
0,TESK1,TESK1_p.His539Tyr,YSLPRAAAL,HLA-C*06:02,0.977559,1.315943,0.601534,1
1,TESK1,TESK1_p.His539Tyr,YSLPRAAAL,HLA-C*06:02,0.977559,1.315943,0.601534,2
2,DDX3X,DDX3X_p.Arg108Thr,STVRPCVVY,HLA-A*29:02,0.948855,2.659422,0.594814,3
3,MED14,MED14_p.Phe1325Leu,LLPDQATQL,HLA-C*06:02,0.954728,0.870841,0.593279,4
4,PRDX5,PRDX5_p.Phe157Leu,LLADPTGAL,HLA-C*06:02,0.763804,0.489961,0.591919,5
5,MED14,MED14_p.Phe224Leu,LLPDQATQL,HLA-C*06:02,0.954728,0.870841,0.591189,6
6,ZNF548,ZNF548_p.Asp24Tyr,TQGRVVFEY,HLA-A*29:02,0.737393,431.709652,0.591101,7
7,ZNF548,ZNF548_p.Asp15Tyr,TQGRVVFEY,HLA-A*29:02,0.737393,431.709652,0.589693,8
8,DDX3X,DDX3X_p.Arg261Thr,STVRPCVVY,HLA-A*29:02,0.948855,2.659422,0.585131,9
9,DDX3X,DDX3X_p.Arg294Thr,STVRPCVVY,HLA-A*29:02,0.948855,2.659422,0.585061,10


## Saving Results

In [7]:
import os

results_dir = Path("../results")
os.makedirs(str(results_dir), exist_ok=True)

out_path = results_dir / "HCC1395_neoantigens.tsv"
results.to_csv(str(out_path))

print(f"Results saved to {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

Results saved to /home/jan/Dropbox/personal_projects/Neoantigen_project/NeoantigenPipeline/results/HCC1395_neoantigens.tsv
File size: 2869.4 KB
Rows: 14240, Columns: 14
